# 12 — EDA & statistics (trial-level features)

Rigorous exploratory analysis of the cached trial feature table
(`outputs/metrics/trial_features.csv`, 348 trials × 25 features + metadata).
All heavy lifting lives in `src.eda` / `src.stats` / `src.config`; this
notebook is a thin driver so the figures in the thesis match the saved
artifacts exactly.

**Statistical choices** (see `src/stats.py`): Mann–Whitney U (non-parametric),
Benjamini–Hochberg FDR (25 simultaneous tests), Cliff's delta effect size,
subject-level bootstrap CIs. Trial-level analysis is exploratory — the
confirmatory metric is the subject-level LOSO baseline in Notebook 13.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
from src import config as C, stats as S, eda

df = C.load_trial_table()
print(f'{len(df)} trials, {df.subject_id.nunique()} subjects, '
      f'{df.group.value_counts().to_dict()}')
df.head()

## 1. Class balance and trials per subject

In [ ]:
print(eda.fig_class_balance(df))
display(Image(str(C.FIGURES / 'eda_class_balance.png')))

## 2. The duration confound (read this before trusting any feature)

PD trials run longer than control trials — strongly in test1 (the chair
stand–up TUG), weakly in test2. Any feature that sums energy over time will
look discriminative for the wrong reason. The figure annotates each test with
Mann–Whitney p and Cliff's delta.

In [ ]:
import json
print(json.dumps(eda.fig_duration_confound(df), indent=2))
display(Image(str(C.FIGURES / 'eda_duration_confound.png')))

## 3. Per-feature group comparison (MWU + BH-FDR + Cliff's delta)

Red bars are features we flagged as duration-confounded. If those dominate the
effect-size ranking, the naive 'most discriminative' features are duration in
disguise.

In [ ]:
print(eda.fig_group_comparison(df))
tbl = pd.read_csv(C.METRICS / 'eda_group_comparison.csv')
display(tbl.head(12))
display(Image(str(C.FIGURES / 'eda_effect_sizes.png')))

## 4. Distributions of the top duration-invariant discriminators

In [ ]:
print(eda.fig_top_feature_distributions(df))
display(Image(str(C.FIGURES / 'eda_top_feature_distributions.png')))

## 5. Correlation structure + duration-leakage audit

The audit shows that several *nominally* duration-invariant features still
correlate with trial length (longer PD trials contain more standing/turning
frames, which shifts per-pixel statistics). Features with |r| < 0.3 with
duration form the `shape_clean` set used in Notebook 13.

In [ ]:
print(json.dumps(eda.fig_correlation_and_duration_audit(df), indent=2))
display(Image(str(C.FIGURES / 'eda_correlation.png')))
display(Image(str(C.FIGURES / 'eda_duration_audit.png')))

## 6. PCA — is there visible separation?

In [ ]:
print(eda.fig_pca(df))
display(Image(str(C.FIGURES / 'eda_pca.png')))

## Takeaways

- Class balance ~57% control / 43% PD at subject level — mild, manageable.
- The strongest trial-level discriminators are duration-confounded energy
  sums; the confound is real in test1 (significant) and weak in test2.
- Even some shape features leak duration — hence the data-driven `shape_clean`
  set and, ultimately, the fixed-length windowing in preprocessing (which
  removes the confound by construction).
- PCA shows no clean linear separation → expect a modest classical baseline
  (confirmed in Notebook 13) and motivates the windowed deep-learning route.

Machine-readable summary: `reports/eda_summary.json`.